### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mice_protein_trisomy_discriminant",
    dataset_year="2015",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C50S3Z",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/342/mice+protein+expression.zip && unzip mice+protein+expression.zip && rm -rf mice+protein+expression.zip
mkdir -p local-data-warehouse/mice_protein_trisomy_discriminant && mv Data_Cortex_Nuclear.xls local-data-warehouse/mice_protein_trisomy_discriminant/
""",
    # References
    academic_reference_bibtex="""@article{higuera2015self,
  title={Self-organizing feature maps identify proteins critical to learning in a mouse model of down syndrome},
  author={Higuera, Clara and Gardiner, Katheleen J and Cios, Krzysztof J},
  journal={PloS one},
  volume={10},
  number={6},
  pages={e0129126},
  year={2015},
  publisher={Public Library of Science San Francisco, CA USA}
}
""",
    academic_reference_bibtex_key="higuera2015self",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We start with the dataset from UCI.

- We drop the three feature that make up the target to remove the leakage.
- We reduce the mouse ID to represent the ID of the mouse instead of the experiment for the mouse.
- We drop the duplicated feature pS6_N.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
    group_on="MouseID",
    group_labels="per_group",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_excel(dataset_mold.path / "Data_Cortex_Nuclear.xls")
print("Loaded data shape:", df.shape)

# Fix the ID so it reduces to the same mouse
df["MouseID"] = df["MouseID"].str.split("_").str[0]
# Drop leaky features that are part of the target
df = df.drop(columns=["Genotype", "Treatment", "Behavior"])
as_type_cat = ["MouseID", "class"]
df[as_type_cat] = df[as_type_cat].astype("category")
# Drop duplicated column
df = df.drop(columns=["pS6_N"])
df = df.reset_index(drop=True)

Loaded data shape: (1080, 82)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,080
Columns: 78
Use sampling: False (sample size: 1,080)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['PSD95_N', 'pCASP9_N', 'Ubiquitin_N', 'S6_N', 'CDK5_N', 'pPKCG_N', 'BAX_N', 'RRP1_N', 'ARC_N', 'pGSK3B_Tyr216_N']
Rows remaining as candidates after top-10 filter: 0 (of 1,080)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,MouseID,DYRK1A_N,ITSN1_N,BDNF_N,NR1_N,NR2A_N,pAKT_N,pBRAF_N,pCAMKII_N,pCREB_N,pELK_N,pERK_N,pJNK_N,PKCA_N,pMEK_N,pNR1_N,pNR2A_N,pNR2B_N,pPKCAB_N,pRSK_N,AKT_N,BRAF_N,CAMKII_N,CREB_N,ELK_N,ERK_N,GSK3B_N,JNK_N,MEK_N,TRKA_N,RSK_N,APP_N,Bcatenin_N,SOD1_N,MTOR_N,P38_N,pMTOR_N,DSCR1_N,AMPKA_N,NR2B_N,pNUMB_N,RAPTOR_N,TIAM1_N,pP70S6_N,NUMB_N,P70S6_N,pGSK3B_N,pPKCG_N,CDK5_N,S6_N,ADARB1_N,AcetylH3K9_N,RRP1_N,BAX_N,ARC_N,ERBB4_N,nNOS_N,Tau_N,GFAP_N,GluR3_N,GluR4_N,IL1B_N,P3525_N,pCASP9_N,PSD95_N,SNCA_N,Ubiquitin_N,pGSK3B_Tyr216_N,SHH_N,BAD_N,BCL2_N,pCFOS_N,SYP_N,H3AcK18_N,EGR1_N,H3MeK4_N,CaNA_N,class
0,309,0.503644,0.747193,0.430175,2.816329,5.990152,0.218830,0.177565,2.373744,0.232224,1.750936,0.687906,0.306382,0.402698,0.296927,1.022060,0.605673,1.877684,2.308745,0.441599,0.859366,0.416289,0.369608,0.178944,1.866358,3.685247,1.537227,0.264526,0.319677,0.813866,0.165846,0.453910,3.037621,0.369510,0.458539,0.335336,0.825192,0.576916,0.448099,0.586271,0.394721,0.339571,0.482864,0.294170,0.182150,0.842725,0.192608,1.443091,0.294700,0.354605,1.339070,0.170119,0.159102,0.188852,0.106305,0.144989,0.176668,0.125190,0.115291,0.228043,0.142756,0.430957,0.247538,1.603310,2.014875,0.108234,1.044979,0.831557,0.188852,0.122652,NaN,0.108336,0.427099,0.114783,0.131790,0.128186,1.675652,c-CS-m
1,309,0.514617,0.689064,0.411770,2.789514,5.685038,0.211636,0.172817,2.292150,0.226972,1.596377,0.695006,0.299051,0.385987,0.281319,0.956676,0.587559,1.725774,2.043037,0.445222,0.834659,0.400364,0.356178,0.173680,1.761047,3.485287,1.509249,0.255727,0.304419,0.780504,0.157194,0.430940,2.921882,0.342279,0.423560,0.324835,0.761718,0.545097,0.420876,0.545097,0.368255,0.321959,0.454519,0.276431,0.182086,0.847615,0.194815,1.439460,0.294060,0.354548,1.306323,0.171427,0.158129,0.184570,0.106592,0.150471,0.178309,0.134275,0.118235,0.238073,0.142037,0.457156,0.257632,1.671738,2.004605,0.109749,1.009883,0.849270,0.200404,0.116682,NaN,0.104315,0.441581,0.111974,0.135103,0.131119,1.743610,c-CS-m
2,309,0.509183,0.730247,0.418309,2.687201,5.622059,0.209011,0.175722,2.283337,0.230247,1.561316,0.677348,0.291276,0.381002,0.281710,1.003635,0.602449,1.731873,2.017984,0.467668,0.814329,0.399847,0.368089,0.173905,1.765544,3.571456,1.501244,0.259614,0.311747,0.785154,0.160895,0.423187,2.944136,0.343696,0.425005,0.324852,0.757031,0.543620,0.404630,0.552994,0.363880,0.313086,0.447197,0.256648,0.184388,0.856166,0.200737,1.524364,0.301881,0.386087,1.279600,0.185456,0.148696,0.190532,0.108303,0.145330,0.176213,0.132560,0.117760,0.244817,0.142445,0.510472,0.255343,1.663550,2.016831,0.108196,0.996848,0.846709,0.193685,0.118508,NaN,0.106219,0.435777,0.111883,0.133362,0.127431,1.926427,c-CS-m
3,309,0.442107,0.617076,0.358626,2.466947,4.979503,0.222886,0.176463,2.152301,0.207004,1.595086,0.583277,0.296729,0.377087,0.313832,0.875390,0.520293,1.566852,2.132754,0.477671,0.727705,0.385639,0.362970,0.179449,1.286277,2.970137,1.419710,0.259536,0.279218,0.734492,0.162210,0.410615,2.500204,0.344509,0.429211,0.330121,0.746980,0.546763,0.386860,0.547849,0.366771,0.328492,0.442650,0.398534,0.161768,0.760234,0.184169,1.612382,0.296382,0.290680,1.198765,0.159799,0.166112,0.185323,0.103184,0.140656,0.163804,0.123210,0.117439,0.234947,0.145068,0.430996,0.251103,1.484624,1.957233,0.119883,0.990225,0.833277,0.192112,0.132781,NaN,0.111262,0.391691,0.130405,0.147444,0.146901,1.700563,c-CS-m
4,309,0.434940,0.617430,0.358802,2.365785,4.718679,0.213106,0.173627,2.134014,0.192158,1.504230,0.550960,0.286961,0.363502,0.277964,0.864912,0.507990,1.480059,2.013697,0.483416,0.687794,0.367531,0.355311,0.174836,1.324695,2.896334,1.359876,0.250705,0.273667,0.702699,0.154827,0.398550,2.456560,0.329126,0.408755,0.313415,0.691956,0.536860,0.360816,0.512824,0.351551,0.312206,0.419095,0.393447,0.160200,0.768113,0.185718,1.645807,0.296829,0.309345,1.206995,0.164650,0.160687,0.188221,0.104784,0.141983,0.167710,0.136838,0.116048,0.255528,0.140871,0.481227,0.251773,1.534835,2.009109,0.119524,0.997775,0.878668,0.205604,0.129954,NaN,0.11

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,MouseID,category,0.0,0.00,72.0,"18899, 293, 294, 309, 311, 320, 321, 322, 3411, 3412"
1,class,category,0.0,0.00,8.0,"c-CS-m, c-SC-m, c-CS-s, c-SC-s, t-CS-m, t-SC-m, t-SC-s, t-CS-s"
2,BCL2_N,float64,285.0,26.39,795.0,"0.1726, 0.1786, 0.1436, 0.1329, 0.1824, 0.1638, 0.1431, 0.2023, 0.1783, 0.1604"
3,H3MeK4_N,float64,270.0,25.00,810.0,"0.3653, 0.1282, 0.1311, 0.1274, 0.1469, 0.1484, 0.1422, 0.1575, 0.1594, 0.1582"
4,BAD_N,float64,213.0,19.72,866.0,"0.2003, 0.1356, 0.1411, 0.1378, 0.1557, 0.1496, 0.1511, 0.1468, 0.1729, 0.1507"
5,EGR1_N,float64,210.0,19.44,870.0,"0.253, 0.1318, 0.1351, 0.1334, 0.1474, 0.1403, 0.1904, 0.1601, 0.1827, 0.1864"
6,H3AcK18_N,float64,180.0,16.67,900.0,"0.3351, 0.1148, 0.112, 0.1119, 0.1651, 0.148, 0.158, 0.163, 0.1714, 0.1724"
7,pCFOS_N,float64,75.0,6.94,1005.0,"0.1876, 0.1083, 0.1043, 0.1062, 0.1113, 0.1107, 0.1094, 0.1115, 0.1131, 0.1055"
8,ELK_N,float64,18.0,1.67,1062.0,"0.8847, 1.8664, 1.761, 1.7655, 1.2863, 1.3247, 1.5303, 1.1515, 1.233, 1.23"
9,Bcatenin_N,float64,18.0,1.67,1062.0,"1.7428, 3.0376, 2.9219, 2.9441, 2.5002, 2.4566, 2.872, 2.4499, 2.5657, 2.5414"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
DYRK1A_N,1077.0,0.425810,0.249362,0.145327,2.516367
ITSN1_N,1077.0,0.617102,0.251640,0.245359,2.602662
BDNF_N,1077.0,0.319088,0.049383,0.115181,0.497160
NR1_N,1077.0,2.297269,0.347293,1.330831,3.757641
NR2A_N,1077.0,3.843934,0.933100,1.737540,8.482553
pAKT_N,1077.0,0.233168,0.041634,0.063236,0.539050
pBRAF_N,1077.0,0.181846,0.027042,0.064043,0.317066
pCAMKII_N,1077.0,3.537109,1.295169,1.343998,7.464070
pCREB_N,1077.0,0.212574,0.032587,0.112812,0.306247
pELK_N,1077.0,1.428682,0.466904,0.429032,6.113347


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column  rank                      
MouseID 1      18899     15   1.39
        2        293     15   1.39
        3        294     15   1.39
        4        309     15   1.39
        5        311     15   1.39
class   1     c-CS-m    150  13.89
        2     c-SC-m    150  13.89
        3     c-CS-s    135  12.50
        4     c-SC-s    135  12.50
        5     t-CS-m    135  12.50

In [8]:
# Target Distribution
target_df

,count,pct
class,,
c-CS-m,150,13.89
c-SC-m,150,13.89
c-CS-s,135,12.50
c-SC-s,135,12.50
t-CS-m,135,12.50
t-SC-m,135,12.50
t-SC-s,135,12.50
t-CS-s,105,9.72


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Providing recommendations based on number of groups (72).
Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 20-repeated 3-fold split. This creates 24 group members (360 samples) per test set.",
    splits=splits
)

Using Stratified Grouped splits.
Using label-per-group grouped splits.
Creating index-based splits for 72 groups
Using Stratified IID splits.
Repeat 0, Fold 0:
            Train N: 720, Test N: 360
            Target Distribution:
            	Train target distribution: {'c-SC-m': 0.14583333333333334, 'c-CS-m': 0.125, 'c-CS-s': 0.125, 'c-SC-s': 0.125, 't-CS-m': 0.125, 't-SC-m': 0.125, 't-SC-s': 0.125, 't-CS-s': 0.10416666666666667}
            	Test target distribution: {'c-CS-m': 0.16666666666666666, 'c-CS-s': 0.125, 'c-SC-m': 0.125, 'c-SC-s': 0.125, 't-CS-m': 0.125, 't-SC-m': 0.125, 't-SC-s': 0.125, 't-CS-s': 0.08333333333333333}
            Group Distribution MouseID:
            	Train: 48
            	Test: 24
            
Repeat 0, Fold 1:
            Train N: 720, Test N: 360
            Target Distribution:
            	Train target distribution: {'c-CS-m': 0.14583333333333334, 'c-CS-s': 0.125, 'c-SC-m': 0.125, 'c-SC-s': 0.125, 't-CS-m': 0.125, 't-SC-m': 0.125, 't-SC-s': 0.125,

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to mice_protein_trisomy_discriminant/019d3198-e47c-7c16-8878-751d864d733a
019d3198-e47c-7c16-8878-751d864d733a
30c64cbd348be8ecf7a4d839a948effc287005d784aafccf75b2f0ef209f25bb
